In [35]:
import numpy as np
import pandas as pd
import uncertainties as un
import uncertainties.unumpy as unp

from uncertainties.unumpy import nominal_values as noms

idx = pd.IndexSlice

## Load Data

In [36]:
experiments = ['Mg', 'Sr', 'Mg+Sr', 'B',]

In [37]:
dat = pd.read_csv('data/data_vaterite_26052022_python.csv', header=[0,1,2,3], index_col=0)
dat.sort_index(axis=1, inplace=True)

mineralogy = pd.read_pickle('data/precipitate_mineralogy.pkl')
dat.loc[mineralogy.index, ('precipitate', 'fraction_vaterite', 'XRD', 'mass_fraction')] = mineralogy['fV']

dat.head()

Type       metadata            precipitate                                     \
Quantity      Batch Experiment        B/Ca    Mg/Ca    Na/Ca    Sr/Ca    d13C   
Instrument       NA         NA        iCap     iCap     iCap     iCap GS-IRMS   
Unit             NA         NA    mmol/mol mmol/mol mmol/mol mmol/mol  permil   
15                1    Control         NaN  -0.0224  26.0077   0.0271  108.01   
16                1         Mg         NaN   0.8879  11.3045   0.0290  116.33   
17                1         Mg         NaN   1.7272  11.1567   0.0294  114.12   
18                1         Mg         NaN   3.8458   8.2277   0.0318  121.50   
19                1         Mg         NaN   6.6334   8.6663   0.0330  136.93   

Type                                                     seed  ...  \
Quantity   diameter percent_vaterite                     B/Ca  ...   
Instrument      SEM              XRD Agilent (10) / iCap (28)  ...   
Unit             μm          percent                 mmol/mol  ...   
15             5.57            79.47                   0.0168  ...   
16             5.60            80.31                   0.0168  ...   
17             5.59            80.99                   0.0168  ...   
18             5.39            79.71                   0.0168  ...   
19             5.19            79.71                   0.0168  ...   

Type       solution_end           solution_start                         \
Quantity           d13C    pH_NBS              B           C         Ca   
Instrument      GS-IRMS electrode        Agilent dissolution    Agilent   
Unit             permil        NA           mmol        mmol       mmol   
15                -2.00     7.461       0.010101           5  12.845237   
16                -2.43     7.486       0.045179           5  12.208096   
17                -3.12     7.479      -0.003059           5  10.702777   
18                -4.12     7.523       0.030233           5  10.712261   
19                -4.28     7.579       0.012196           5  11.587594   

Type                                                       precipitate  
Quantity          Mg          Na        Sr    pH_NBS fraction_vaterite  
Instrument   Agilent     Agilent   Agilent electrode               XRD  
Unit            mmol        mmol      mmol        NA     mass_fraction  
15         -0.001428  247.976348  0.001013     8.213   0.7784+/-0.0025  
16          1.026357  224.830172  0.000964     8.209   0.8010+/-0.0023  
17          2.068886  233.039178  0.000869     8.193   0.8073+/-0.0022  
18          4.928338  223.381524  0.000888     8.200   0.7945+/-0.0023  
19          9.428138  213.820447  0.001118     8.198   0.7646+/-0.0026  

[5 rows x 31 columns]

### Set negative measured compositions to mean for their group

In [38]:
for exp in experiments:
    sections = ['precipitate', 'seed']
    vars = ['Mg/Ca', 'Sr/Ca', 'Na/Ca', 'B/Ca']
    expind = (dat.metadata.Experiment == exp).squeeze()
    for sec in sections:
        for var in vars:
            ind = (dat[(sec, var)] < 0).squeeze() & expind
            dat.loc[ind, (sec,var)] = dat.loc[expind, (sec,var)].mean().item()
            
    sections = ['solution_start', 'solution_end']
    vars = ['B', 'Ca', 'Mg', 'Na', 'Sr']

    for sec in sections:
        for var in vars:
            ind = (dat[(sec, var)] < 0).squeeze() & expind
            dat.loc[ind, (sec,var)] = dat.loc[expind, (sec,var)].mean().item()

/tmp/ipykernel_217497/3395733201.py:7: PerformanceWarning: indexing past lexsort depth may impact performance.
  ind = (dat[(sec, var)] < 0).squeeze() & expind
/tmp/ipykernel_217497/3395733201.py:8: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat.loc[ind, (sec,var)] = dat.loc[expind, (sec,var)].mean().item()
/tmp/ipykernel_217497/3395733201.py:7: PerformanceWarning: indexing past lexsort depth may impact performance.
  ind = (dat[(sec, var)] < 0).squeeze() & expind
/tmp/ipykernel_217497/3395733201.py:8: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat.loc[ind, (sec,var)] = dat.loc[expind, (sec,var)].mean().item()
/tmp/ipykernel_217497/3395733201.py:7: PerformanceWarning: indexing past lexsort depth may impact performance.
  ind = (dat[(sec, var)] < 0).squeeze() & expind
/tmp/ipykernel_217497/3395733201.py:8: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat.loc[ind, (sec,var)] = dat.loc[expind, (sec,

### Package uncertainties

In [39]:
composition_Fstd = 0.05  # 5% on compositions

isotope_std = 1.  # 1 permil on C isotopes (outside SRM range)
xrd_std = 0.02  # 2% on xrd

In [40]:
ind = idx[:,:,['iCap', 'Agilent', 'Agilent (10) / iCap (28)'],:]
cols = dat.loc[:,ind].columns
dat[cols] = dat[cols].astype(object)

dat.loc[:,ind] = unp.uarray(
    nominal_values=dat.loc[:,ind].values,
    std_devs=abs(dat.loc[:,ind].values) * composition_Fstd
)

ind = idx[:,:,'GS-IRMS',:]
cols = dat.loc[:,ind].columns
dat[cols] = dat[cols].astype(object)

dat.loc[:,ind] = unp.uarray(
    nominal_values=dat.loc[:,ind].values,
    std_devs=isotope_std
)

# ind = idx[:,:,'XRD',:]
# dat.loc[:,ind] = unp.uarray(
#     nominal_values=dat.loc[:,ind].values,
#     std_devs=xrd_std
# )

/home/oscar/.venvs/py3/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


### Calculate solution ratios

In [41]:
for el in ['Mg', 'Na', 'Sr']:
    dat[('solution_start', f'{el}/Ca', 'calc', 'mol/mol')] = (
        dat[('solution_start', f'{el}')].squeeze() / 
        dat[('solution_start', 'Ca')].squeeze()
    )
    dat[('solution_end', f'{el}/Ca', 'calc', 'mol/mol')] = (
        dat[('solution_end', f'{el}')].squeeze() / 
        dat[('solution_end', 'Ca')].squeeze()
    )

dat[('solution_start', 'B/C', 'calc', 'mol/mol')] = (
    dat[('solution_start', 'B')].squeeze() / 
    dat[('solution_start', 'C')].squeeze()
)

/tmp/ipykernel_217497/1850766334.py:3: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat[('solution_start', f'{el}')].squeeze() /
/tmp/ipykernel_217497/1850766334.py:4: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat[('solution_start', 'Ca')].squeeze()
/tmp/ipykernel_217497/1850766334.py:7: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat[('solution_end', f'{el}')].squeeze() /
/tmp/ipykernel_217497/1850766334.py:8: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat[('solution_end', 'Ca')].squeeze()
/tmp/ipykernel_217497/1850766334.py:12: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat[('solution_start', 'B')].squeeze() /
/tmp/ipykernel_217497/1850766334.py:13: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat[('solution_start', 'C')].squeeze()


### 1. Calculate the fractional abundance of overgrowth in the experiment

$$
\begin{align}
f &= \frac{M_{OG}}{M_{OG} + M{seed}} \\
\delta^{13}C_{PPT} &= \delta^{13}C_{seed} (1 - f) + \delta^{13}C_{OG} f \\
\delta^{13}C_{PPT} &= \delta^{13}C_{seed} - f (\delta^{13}C_{seed} - \delta^{13}C_{OG}) \\
f &= \frac{\delta^{13}C_{seed} - \delta^{13}C_{PPT}}{\delta^{13}C_{seed} - \delta^{13}C_{OG}}
\end{align}
$$

In [42]:
d13C_OG_mean = noms(dat.loc[(dat.metadata == 1).values, ('solution_end', 'd13C')]).mean().item()
d13C_OG_std = noms(dat.loc[(dat.metadata == 1).values, ('solution_end', 'd13C')]).std().item()

d13C_OG = un.ufloat(d13C_OG_mean, d13C_OG_std)

d13C_OG

/tmp/ipykernel_217497/4210286498.py:1: PerformanceWarning: indexing past lexsort depth may impact performance.
  d13C_OG_mean = noms(dat.loc[(dat.metadata == 1).values, ('solution_end', 'd13C')]).mean().item()
/tmp/ipykernel_217497/4210286498.py:2: PerformanceWarning: indexing past lexsort depth may impact performance.
  d13C_OG_std = noms(dat.loc[(dat.metadata == 1).values, ('solution_end', 'd13C')]).std().item()


-3.6814285714285715+/-1.0893435581831392

In [43]:
dat[('derived', 'f', 'calc', 'fractional_abundance')] = (
    (dat.loc[:, ('seed','d13C')] - dat.loc[:, ('precipitate','d13C')]) / 
    (dat.loc[:, ('seed','d13C')] - d13C_OG)
)

/tmp/ipykernel_217497/732595139.py:2: PerformanceWarning: indexing past lexsort depth may impact performance.
  (dat.loc[:, ('seed','d13C')] - dat.loc[:, ('precipitate','d13C')]) /
/tmp/ipykernel_217497/732595139.py:3: PerformanceWarning: indexing past lexsort depth may impact performance.
  (dat.loc[:, ('seed','d13C')] - d13C_OG)


In [44]:
dat.derived.f.head()

Instrument,calc
Unit,fractional_abundance
15,0.435+/-0.006
16,0.393+/-0.006
17,0.404+/-0.006
18,0.366+/-0.006
19,0.288+/-0.006


### 2. Calculate Overgrowth composition

$$
\begin{align}
X/Ca_{PPT} &= X/Ca_{seed} (1 - f)  + X/Ca_{OG} f \\
X/Ca_{OG} &= \frac{X/Ca_{PPT} - X/Ca_{seed}(1 - f)}{f}  \\
\end{align}
$$

In [45]:
ratios = ['Mg/Ca', 'Sr/Ca', 'Na/Ca', 'B/Ca']

for rat in ratios:
    dat[('overgrowth', rat, 'calc', 'mol/mol')] = (
        1e-3 * (
            dat[('precipitate', rat)].squeeze() - 
            (1 - dat.derived.f.squeeze()) * 
            dat[('seed', rat)].squeeze()
            ) / 
        dat.derived.f.squeeze()
    )
    # # replace any values below zero with zeros
    # ind = dat[('overgrowth', rat, 'calc', 'mol/mol')] < 0
    # dat.loc[ind, ('overgrowth', rat, 'calc', 'mol/mol')] = 0

/tmp/ipykernel_217497/1480119082.py:6: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat[('precipitate', rat)].squeeze() -
/tmp/ipykernel_217497/1480119082.py:8: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat[('seed', rat)].squeeze()


#### Calculation overgrowth partitioning 

In [46]:
for el in ['Mg', 'Na', 'Sr']:
    dat[('overgrowth', f'D_{el}', 'calc', None)] = (
        dat[('overgrowth', f'{el}/Ca')] / 
        dat[('solution_start', f'{el}/Ca')]
    )

dat[('overgrowth', f'D_B', 'calc', None)] = (
    dat[('overgrowth', 'B/Ca')] / 
    dat[('solution_start', f'B/C')]
)

/tmp/ipykernel_217497/3210602181.py:3: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat[('overgrowth', f'{el}/Ca')] /
/tmp/ipykernel_217497/3210602181.py:4: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat[('solution_start', f'{el}/Ca')]
/tmp/ipykernel_217497/3210602181.py:8: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat[('overgrowth', 'B/Ca')] /
/tmp/ipykernel_217497/3210602181.py:9: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat[('solution_start', f'B/C')]


In [47]:
dat.overgrowth.head()

Quantity,Mg/Ca,Sr/Ca,Na/Ca,B/Ca,D_Mg,D_Na,D_Sr,D_B
Instrument,calc,calc,calc,calc,calc,calc,calc,calc
Unit,mol/mol,mol/mol,mol/mol,mol/mol,nan,NaN,NaN,NaN
15,-0.000107+/-0.000004,(-6+/-5)e-06,0.0449+/-0.0031,nan+/-nan,0.96+/-0.08,0.00232+/-0.00023,-0.08+/-0.06,nan+/-nan
16,0.00220+/-0.00012,(-8+/-6)e-06,0.0110+/-0.0017,nan+/-nan,0.0261+/-0.0023,0.00060+/-0.00010,-0.10+/-0.07,nan+/-nan
17,0.00421+/-0.00022,(-5+/-5)e-06,0.0106+/-0.0016,nan+/-nan,0.0218+/-0.0019,0.00049+/-0.00008,-0.06+/-0.07,nan+/-nan
18,0.0104+/-0.0006,(-4+/-6)e-06,0.0026+/-0.0015,nan+/-nan,0.0226+/-0.0020,0.00012+/-0.00007,-0.05+/-0.08,nan+/-nan
19,0.0229+/-0.0013,(-1.5+/-0.9)e-05,0.0016+/-0.0021,nan+/-nan,0.0281+/-0.0025,0.00009+/-0.00011,-0.16+/-0.09,nan+/-nan


### 3. Calculate the fraction of vaterite in the overgrowth

$$
\begin{align}
F_{V,OG} &= \frac{M_{V,OG}}{M_{V,OG} + M_{C,OG}} \\
F_{V,ppt} &= (1 - f) F_{V,seed} + f F_{V,OG} \\
F_{V,OG} &= \frac{F_{V,ppt} - (1 - f) F_{V,seed}}{f}
\end{align}
$$

Assuming this does not affect the geometry of the overgrowth (approx. spherical).

In [48]:
dat.derived.f

Instrument,calc
Unit,fractional_abundance
15,0.435+/-0.006
16,0.393+/-0.006
17,0.404+/-0.006
18,0.366+/-0.006
19,0.288+/-0.006
20,0.243+/-0.006
21,0.261+/-0.006
22,0.371+/-0.006
23,0.433+/-0.006


In [49]:
dat[('overgrowth', 'F_V', 'calc', 'fractional_abundance')] = ((dat.precipitate.fraction_vaterite.squeeze()) - (dat.seed.percent_vaterite.squeeze() * (1 - dat.derived.f.squeeze()) / 100)) / dat.derived.f.squeeze()
dat[('overgrowth', 'F_V_percent', 'calc', 'fractional_abundance')] = ((dat.precipitate.percent_vaterite.squeeze() / 100) - (dat.seed.percent_vaterite.squeeze() * (1 - dat.derived.f.squeeze()) / 100)) / dat.derived.f.squeeze()

In [50]:
dat.overgrowth.head(6)

Quantity,Mg/Ca,Sr/Ca,Na/Ca,B/Ca,D_Mg,D_Na,D_Sr,D_B,F_V,F_V_percent
Instrument,calc,calc,calc,calc,calc,calc,calc,calc,calc,calc
Unit,mol/mol,mol/mol,mol/mol,mol/mol,nan,NaN,NaN,NaN,fractional_abundance,fractional_abundance
15,-0.000107+/-0.000004,(-6+/-5)e-06,0.0449+/-0.0031,nan+/-nan,0.96+/-0.08,0.00232+/-0.00023,-0.08+/-0.06,nan+/-nan,0.551+/-0.008,0.589+/-0.005
16,0.00220+/-0.00012,(-8+/-6)e-06,0.0110+/-0.0017,nan+/-nan,0.0261+/-0.0023,0.00060+/-0.00010,-0.10+/-0.07,nan+/-nan,0.566+/-0.008,0.571+/-0.006
17,0.00421+/-0.00022,(-5+/-5)e-06,0.0106+/-0.0016,nan+/-nan,0.0218+/-0.0019,0.00049+/-0.00008,-0.06+/-0.07,nan+/-nan,0.592+/-0.008,0.598+/-0.006
18,0.0104+/-0.0006,(-4+/-6)e-06,0.0026+/-0.0015,nan+/-nan,0.0226+/-0.0020,0.00012+/-0.00007,-0.05+/-0.08,nan+/-nan,0.520+/-0.010,0.527+/-0.007
19,0.0229+/-0.0013,(-1.5+/-0.9)e-05,0.0016+/-0.0021,nan+/-nan,0.0281+/-0.0025,0.00009+/-0.00011,-0.16+/-0.09,nan+/-nan,0.299+/-0.017,0.412+/-0.012
20,0.0336+/-0.0019,(0.5+/-1.2)e-05,0.0017+/-0.0026,nan+/-nan,0.0182+/-0.0016,0.00008+/-0.00012,0.06+/-0.14,nan+/-nan,0.533+/-0.013,0.343+/-0.016


### 4. Estimate the composition of the vaterite from inorganic partition coefficients

$$
\begin{align}
D_{OG} &= D_C (1 - F_{V,OG})  + D_V F_{V,OG} \\
D_V &= \frac{D_{OG} - D_C (1 - F_{V,OG})}{F_{V,OG}}
\end{align}
$$

$$
D = \frac{X/Ca_{solid}}{X/Ca_{fluid}}
$$

In [51]:
# All *very* approximate
D_calcite = {
    'Mg': un.ufloat(0.014, 0.02),  # DMg from Mucci & Morse (1983) via Hasiuk 2010 (10.1016/j.gca.2010.07.030)
    'Sr': un.ufloat(7e-2, 2e-2),  # DSr from Morse & Bender (1990)
    'Na': un.ufloat(1e-4, 0.1e-4),  # DNa from Kawabata et al (2021; 10.1016/j.chemgeo.2020.119904)
    'B': un.ufloat(1e-3, 0.2e-3),  # DB (BT/DIC) from Uchikawa et al (2015)
}

for el, D_cal in D_calcite.items():
    
    dat[('vaterite', f'D_{el}', 'calc', None)] = (dat[('overgrowth', f'D_{el}')].squeeze() - (1 - dat[('overgrowth', 'F_V')].squeeze()) * D_cal) / dat[('overgrowth', 'F_V')].squeeze()

    if el != 'B':
        dat[('calcite', f'{el}/Ca', 'calc', None)] = dat[('solution_start', f'{el}/Ca')].squeeze() * D_cal
        dat[('vaterite', f'{el}/Ca', 'calc', 'mol/mol')] = dat[('vaterite', f'D_{el}')].squeeze() * dat[('solution_start', f'{el}/Ca')].squeeze()
    else:
        dat[('vaterite', f'{el}/Ca', 'calc', 'mol/mol')] = dat[('vaterite', f'D_{el}')].squeeze() * dat[('solution_start', f'{el}/C')].squeeze()
        dat[('calcite', f'{el}/Ca', 'calc', None)] = dat[('solution_start', f'{el}/C')].squeeze() * D_cal

/tmp/ipykernel_217497/4227336704.py:11: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat[('vaterite', f'D_{el}', 'calc', None)] = (dat[('overgrowth', f'D_{el}')].squeeze() - (1 - dat[('overgrowth', 'F_V')].squeeze()) * D_cal) / dat[('overgrowth', 'F_V')].squeeze()
/tmp/ipykernel_217497/4227336704.py:14: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat[('calcite', f'{el}/Ca', 'calc', None)] = dat[('solution_start', f'{el}/Ca')].squeeze() * D_cal
/tmp/ipykernel_217497/4227336704.py:15: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat[('vaterite', f'{el}/Ca', 'calc', 'mol/mol')] = dat[('vaterite', f'D_{el}')].squeeze() * dat[('solution_start', f'{el}/Ca')].squeeze()
/tmp/ipykernel_217497/4227336704.py:17: PerformanceWarning: indexing past lexsort depth may impact performance.
  dat[('vaterite', f'{el}/Ca', 'calc', 'mol/mol')] = dat[('vaterite', f'D_{el}')].squeeze() * dat[('solution_start', f'{el}/C')].

### 5. Export data for use in other notebooks

In [52]:
dat.sort_index(axis=0, inplace=True)
dat.sort_index(axis=1, inplace=True)

In [53]:
dat.to_csv('data/vaterite_processed.csv')
dat.to_pickle('data/vaterite_processed.pkl')